In [17]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load Titanic dataset from seaborn (built-in)
import seaborn as sns
df = sns.load_dataset('titanic')

# Show first 5 rows
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [18]:
#Basic Data Preprocessing
# Drop columns that are not useful
df = df.drop(['deck', 'embark_town', 'alive', 'who', 'class'], axis=1)

# Fill missing values
df['age'] = df['age'].fillna(df['age'].median())
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

# Convert categorical features to numeric
df['sex'] = df['sex'].map({'female': 0, 'male': 1})
df = pd.get_dummies(df, columns=['embarked'], drop_first=True)

# Convert boolean columns to int
df['adult_male'] = df['adult_male'].astype(int)
df['alone'] = df['alone'].astype(int)

# Check data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   survived    891 non-null    int64  
 1   pclass      891 non-null    int64  
 2   sex         891 non-null    int64  
 3   age         891 non-null    float64
 4   sibsp       891 non-null    int64  
 5   parch       891 non-null    int64  
 6   fare        891 non-null    float64
 7   adult_male  891 non-null    int64  
 8   alone       891 non-null    int64  
 9   embarked_Q  891 non-null    bool   
 10  embarked_S  891 non-null    bool   
dtypes: bool(2), float64(2), int64(7)
memory usage: 64.5 KB


In [19]:
#Split Dataset
# Features (X) and target (y)
X = df.drop('survived', axis=1)
y = df['survived']

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [20]:
#Feature Engineering
#A. Polynomial & Interaction Features
# Square features
X_train['age2'] = X_train['age'] ** 2
X_train['fare2'] = X_train['fare'] ** 2

# Interaction features
X_train['age_fare'] = X_train['age'] * X_train['fare']
X_train['sibsp_parch'] = X_train['sibsp'] * X_train['parch']

In [21]:
#B. Binning Features
# Age groups
X_train['age_group'] = pd.cut(X_train['age'], bins=[0,12,18,35,60,100], labels=[0,1,2,3,4])

# Fare groups
X_train['fare_group'] = pd.qcut(X_train['fare'], q=4, labels=[0,1,2,3])

In [22]:
#Apply Same Feature Engineering on Test Set
X_test['age2'] = X_test['age'] ** 2
X_test['fare2'] = X_test['fare'] ** 2
X_test['age_fare'] = X_test['age'] * X_test['fare']
X_test['sibsp_parch'] = X_test['sibsp'] * X_test['parch']

X_test['age_group'] = pd.cut(X_test['age'], bins=[0,12,18,35,60,100], labels=[0,1,2,3,4])
X_test['fare_group'] = pd.qcut(X_test['fare'], q=4, labels=[0,1,2,3])

In [24]:
#Feature Selection Using Random Forest
from xgboost import XGBClassifier

# Train XGBoost to get feature importance
xgb = XGBClassifier(eval_metric='logloss', use_label_encoder=False,enable_categorical=True )
xgb.fit(X_train, y_train)

# Feature importance
importances = pd.Series(xgb.feature_importances_, index=X_train.columns)
importances.sort_values(ascending=False)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [11:56:40] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,0
adult_male,0.783576
pclass,0.090765
sibsp_parch,0.017064
age_group,0.014536
fare,0.013060
fare_group,0.010447
sibsp,0.010308
embarked_S,0.010131
age,0.009983
parch,0.009538


In [25]:
#7: Train Model on Engineered Features
# Select top features (for simplicity, take all for now)
X_train_fe = X_train
X_test_fe = X_test

xgb.fit(X_train_fe, y_train)
y_pred = xgb.predict(X_test_fe)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [12:00:03] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [26]:
#Evaluate Performance
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_pred))

Accuracy: 0.7988826815642458
Precision: 0.7435897435897436
Recall: 0.7837837837837838
F1 Score: 0.7631578947368421
ROC-AUC: 0.7966537966537967
